In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("trainloan").getOrCreate()
# Initialize Spark session
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/13 22:57:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


* **Gender**: Male, Female.
* **Married**: Marital status of the applicant.
* **Dependents**: Number of dependents.
* **Education**: Applicant’s education level.
* **Self_Employed**: Whether the Applicant is self-employed or not. 
* **ApplicantIncome**: Income of the applicant.
* **CoapplicantIncome**: Income of the co-applicant.
* **LoanAmoun**t: The loan amount requested.
* **Loan_Amount_Term**: Term of the loan.
* **Credit_History**: Whether the applicant has a credit history (1 = Yes, 0 = No).
* **Property_Area**: Urban, Semiurban, or Rural.
* **Loan_Status** (Target): 1 = Loan approved, 0 = Loan not approved.


In [2]:
#loading file
Loan_df = spark.read.csv("/kaggle/input/trainloan/train_loan.csv", header=True, inferSchema=True)

Loan_df.show(30)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|              0.0|      NULL|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|           1508.0|       128|             360|             1|        Rural|          N|
|LP001005|  Male|    Yes|         0|    Graduate|          Yes|           3000|              0.0|        66|             360|             1|        Urban|          Y

In [3]:
Loan_df.printSchema()

root
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: integer (nullable = true)
 |-- Loan_Amount_Term: integer (nullable = true)
 |-- Credit_History: integer (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)



In [4]:
#checking missing values 
from pyspark.sql.functions import isnan , count, when
from pyspark.sql.functions import col , sum

Loan_df.select([count(when(isnan(c) | col(c).isNull(), c)) for c 
in Loan_df.columns]).show()


+-----------------------------------------------------------------------+--------------------------------------------------------------------+-----------------------------------------------------------------------+--------------------------------------------------------------------------------+-----------------------------------------------------------------------------+-----------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------+------------------------------------------------------------------

* Gender	13
* Married	3
* Dependents	15
* **ApplicantIncome**	32
* **LoanAmount**	22
* **Loan_Amount_Term**	14
* **Credit_History**	50

In [5]:
#fill the missing data with median
#all the new cleaned columns need to be into one dataframe
import pyspark.sql.functions as F

# Calculate all medians
median_applicant_income =Loan_df.agg(F.median('ApplicantIncome')).collect()[0][0]
median_loan_amount =Loan_df.agg(F.median('LoanAmount')).collect()[0][0]
median_loan_term = Loan_df.agg(F.median('Loan_Amount_Term')).collect()[0][0]
median_credit_history = Loan_df.agg(F.median('Credit_History')).collect()[0][0]


                                
# imputation map
imputation_map = {
    "ApplicantIncome" : median_applicant_income,
    "Loan_Amount_Term": median_loan_term,
    "LoanAmount"      : median_loan_amount,
    "Credit_History"  : median_credit_history
}

#fills in a single step
Loan_df_clean = Loan_df.na.fill(imputation_map)

Loan_df_clean.show(5)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|              0.0|       128|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|           1508.0|       128|             360|             1|        Rural|          N|
|LP001005|  Male|    Yes|         0|    Graduate|          Yes|           3000|              0.0|        66|             360|             1|        Urban|          Y

In [6]:
from pyspark.sql import functions as F

# Get modes for all categorical columns in one go
categorical_cols = ['Gender', 'Married', 'Dependents']

modes = {}
for col in categorical_cols:
    mode_value = (Loan_df_clean.groupBy(col).count().orderBy(F.desc('count')).first()[0])
    modes[col] = mode_value

# Apply all categorical imputations
Loan_df_clean = Loan_df_clean.na.fill(modes)
Loan_df_clean.show(5)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|              0.0|       128|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|           1508.0|       128|             360|             1|        Rural|          N|
|LP001005|  Male|    Yes|         0|    Graduate|          Yes|           3000|              0.0|        66|             360|             1|        Urban|          Y

In [7]:
from pyspark.ml.feature import VectorAssembler,StringIndexer
from pyspark.ml import Pipeline

#defines all fetures 
cat_feat= ["Gender","Education","Property_Area","Married","Dependents","Self_Employed"]
num_feat= ["ApplicantIncome","CoapplicantIncome","LoanAmount","Loan_Amount_Term","Credit_History"]

#encodes class label
label_indexer= StringIndexer(inputCol="Loan_Status", outputCol="label")
cl_df = label_indexer.fit(Loan_df_clean).transform(Loan_df_clean)

#encoding categorical data

gender_indexer = StringIndexer(inputCol="Gender", outputCol="gender_label")
education_indexer = StringIndexer(inputCol="Education", outputCol="education_label")
property_indexer = StringIndexer(inputCol="Property_Area", outputCol ="property_label")
married_indexer =StringIndexer(inputCol="Married", outputCol ="married_label")
dependents_indexer = StringIndexer(inputCol= "Dependents", outputCol= "dependents_label")
self_employed_indexer = StringIndexer(inputCol="Self_Employed", outputCol = "self_employed_label")


#defines all categorical indexed features 
cat_indexed_feat= ["gender_label","education_label","property_label","married_label","dependents_label","self_employed_label"]

all_feat= cat_indexed_feat + num_feat


#Assembling features

feat_assembler = VectorAssembler(inputCols=all_feat, outputCol ="features")

# Build pipeline
pipeline = Pipeline(stages=[gender_indexer,education_indexer ,property_indexer, 
married_indexer,dependents_indexer,self_employed_indexer, feat_assembler])

#Fit and transform the data
prep_df= pipeline.fit(cl_df).transform(cl_df).select("features","label")
Loan_df_clean.show(5)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|LP001002|  Male|     No|         0|    Graduate|           No|           5849|              0.0|       128|             360|             1|        Urban|          Y|
|LP001003|  Male|    Yes|         1|    Graduate|           No|           4583|           1508.0|       128|             360|             1|        Rural|          N|
|LP001005|  Male|    Yes|         0|    Graduate|          Yes|           3000|              0.0|        66|             360|             1|        Urban|          Y

In [8]:
#feature